# 2. Simple CSV RAG

RAG over tabular data: turn every row into a self-describing sentence and index it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A lot of business knowledge lives in spreadsheets. Embedding raw CSV text (`P001,AeroBook 14,Laptop,1199...`) works poorly because the model has no idea what each number means. The trick is to turn each row into a **self-describing sentence** so the embedding captures both the value and its column.

## How it works

1. Read the CSV with pandas
2. Serialise each row as `column: value, column: value...`
3. Keep the original row as metadata so you can show or filter it later
4. Embed and index the sentences
5. Retrieve rows and let the LLM read them

```text
 products.csv --> row -> "name: AeroBook 14, price_usd: 1199, ..." --> embed --> store
 question --> embed --> top rows --> LLM --> answer  (+ metadata: the raw row)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** pandas

In [1]:
import sys
sys.path.insert(0, "..")
from common import *
import pandas as pd

df = pd.read_csv(DATA / "products.csv")
df.head()

,product_id,name,category,price_usd,rating,in_stock,description
0,P001,AeroBook 14,Laptop,1199,4.6,yes,Lightweight 14 inch laptop with 18 hour batter...
1,P002,AeroBook 16 Pro,Laptop,1899,4.8,yes,16 inch creator laptop with dedicated GPU and ...
2,P003,PixelBuds Mini,Audio,79,4.2,yes,Compact wireless earbuds with 6 hour battery a...
3,P004,PixelBuds ANC,Audio,149,4.5,no,Wireless earbuds with active noise cancellatio...
4,P005,StudioMonitor 27,Monitor,449,4.7,yes,27 inch 4K monitor with 99 percent sRGB covera...


Each row becomes one document. Keeping the column names in the text lets the embedding capture *what* each value means.

In [2]:
rows = [", ".join(f"{col}: {val}" for col, val in row.items()) for _, row in df.iterrows()]
meta = [row.to_dict() for _, row in df.iterrows()]
store = build_store(rows, meta)
print(rows[0])

W1028 10:23:54.828000 6682 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


product_id: P001, name: AeroBook 14, category: Laptop, price_usd: 1199, rating: 4.6, in_stock: yes, description: Lightweight 14 inch laptop with 18 hour battery and 16GB RAM


In [3]:
for q in ["Which laptop has the longest battery life?", "Recommend a cheap wireless audio product that is in stock", "What is the rating of the StudioMonitor 27?"]:
    response, hits = rag(q, store, k=3)
    print("Q:", q, "\nA:", response, "\n")

Q: Which laptop has the longest battery life? 
A: According to the context, the AeroBook 14 has an 18 hour battery life, which is longer than the AeroBook 16 Pro's battery life (no specified duration). Therefore, the answer is the AeroBook 14. 



Q: Recommend a cheap wireless audio product that is in stock 
A: Based on the provided context, I would recommend the PixelBuds Mini (product_id P003). It is a compact wireless earbud with 6 hour battery life and noise isolation, priced at $79. Since it is mentioned as "in_stock: yes", it meets your criteria of being both cheap and available. 

Q: What is the rating of the StudioMonitor 27? 
A: The rating of the StudioMonitor 27 is 4.7. 



**Limits:** semantic search cannot do aggregation (e.g. *average price*). For that, let the LLM write a pandas/SQL query instead.

## Why the serialisation format matters
Compare a bare CSV line with a labelled one. The labelled form carries meaning for the embedder and for the LLM reading the prompt.

In [4]:
bare = ",".join(str(v) for v in df.iloc[0].tolist())
labelled = rows[0]
print("bare    :", bare)
print("labelled:", labelled)

bare    : P001,AeroBook 14,Laptop,1199,4.6,yes,Lightweight 14 inch laptop with 18 hour battery and 16GB RAM
labelled: product_id: P001, name: AeroBook 14, category: Laptop, price_usd: 1199, rating: 4.6, in_stock: yes, description: Lightweight 14 inch laptop with 18 hour battery and 16GB RAM


## Using row metadata for exact filtering
Semantic search is fuzzy; structured columns are exact. Combine them: filter on the metadata first, then rank the survivors semantically.

In [5]:
import numpy as np
mask = np.array([m["category"] == "Audio" and m["in_stock"] == "yes" for m in meta])
for h in store.search("wireless earbuds with long battery", k=5, mask=mask):
    print(f"{h['score']:.3f}", h["meta"]["name"], h["meta"]["price_usd"])

0.540 PixelBuds Mini 79


## Where RAG over CSV breaks: aggregation
Questions like *average price* or *how many products are in stock* need computation over **all** rows, not the top-k. Observe the failure, then fix it by letting the LLM write pandas code.

In [6]:
q = "What is the average price of all laptops?"
print("RAG answer :", rag(q, store, k=3)[0])
print("Ground truth:", df[df.category == "Laptop"].price_usd.mean())

RAG answer : To find the average price of all laptops, we need to add up their prices and divide by the number of laptops.

The prices of the two laptops are:
- AeroBook 14: $1199
- AeroBook 16 Pro: $1899

Total price = $1199 + $1899 = $3098

There are 2 laptops. 

Average price = Total price / Number of laptops 
= $3098 / 2 
= $1549
Ground truth: 1549.0


In [7]:
def ask_dataframe(question):
    code = llm(
        "Write ONE line of pandas code that answers the question using a DataFrame named df "
        f"with columns {list(df.columns)}. Reply with code only, no markdown.\n\nQuestion: {question}"
    ).strip().strip("`").replace("python", "").strip()
    print("generated:", code)
    return eval(code, {"df": df, "pd": pd})   # demo only: never eval untrusted model output in production

print(ask_dataframe("What is the average price of all laptops?"))

generated: df[df['category'] == 'laptop']['price_usd'].mean()
nan


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Easy: no special infrastructure | Cannot aggregate (sum, average, count) - top-k sees only a few rows |
| Row metadata enables exact filters (stock, category, price) | Wide tables produce long, noisy sentences |
| Labelled text makes both the embedder and LLM more accurate | Numbers are weakly represented by embeddings (is 1199 close to 1299?) |

## When to use it

Lookup-style questions over rows (what is the spec of X, which product fits Y). For analytics, route the question to text-to-pandas or text-to-SQL instead.

## Experiments to try

- Add a `price_usd < 100` filter using the metadata mask.
- Index only `name + description` and compare answers with indexing all columns.
- Route automatically: classify the question as lookup vs aggregation and call the right tool.

## Key takeaways

- Serialise rows with column names.
- Keep structured metadata next to the vectors.
- Semantic search is for meaning; use code/SQL for maths.

---
**Next:** [3. JSON RAG](../03_json_rag/json_rag.ipynb)